# Data Cleaning

In [ ]:
## Import Libraries

import pandas as pd
import numpy as np

In [17]:
listings=pd.read_csv("../data/raw/listings 2.csv")
listings_clean=listings.copy()

print(listings_clean.shape)


(30259, 90)


In [18]:
## Missing value Report
missing_report = pd.DataFrame({
    "Missing Values": listings_clean.isnull().sum(),
    "Missing Percentage": (listings_clean.isnull().sum() /len(listings_clean)).round(4) * 100
})

missing_report.sort_values(
    by="Missing Percentage",
    ascending=False
)

,Missing Values,Missing Percentage
neighbourhood,30259,100.0
host_since,30259,100.0
host_neighbourhood,30259,100.0
host_thumbnail_url,30259,100.0
host_acceptance_rate,30259,100.0
...,...,...
maximum_nights_avg_ntm,0,0.0
availability_30,0,0.0
availability_60,0,0.0
availability_90,0,0.0


# Cleaning Strategy

| Column | Missing % | Action | Reason |
|--------|----------:|--------|--------|
| neighbourhood | 100% | Drop | Completely empty; provides no analytical value. |
| host_since | 100% | Drop | No usable data available. |
| host_neighbourhood | 100% | Drop | Completely missing across all records. |
| host_thumbnail_url | 100% | Drop | Image URL not required for analysis and contains no data. |
| host_acceptance_rate | 100% | Drop | No usable information available. |
| license | High | Keep | May be useful for compliance analysis; not required for current business questions. |
| reviews_per_month | Partial | Fill with 0 | Missing values indicate listings with no reviews. |
| last_review | Partial | Keep as NULL | Missing values indicate listings have never been reviewed. |
| first_review | Partial | Keep as NULL | Missing values indicate no review history. |
| host_response_rate | Partial | Convert to numeric | Remove '%' symbol and convert to numeric for analysis. |
| host_response_time | Partial | Keep | Useful categorical feature for host performance analysis. |
| price | Partial | Convert to numeric | Remove '$' and ',' if present for calculations. |
| bathrooms_text | None | Extract numeric value later | Required for quantitative analysis. |
| amenities | None | Keep | Valuable feature for future analysis and feature engineering. |
| Duplicate Rows | - | Remove (if any) | Prevent duplicate records from affecting analysis. |



Some of the columns are being dropped as the do not contribute any analytical value

In [ ]:
## remove duplicate records

duplicate_rows = listings_clean.duplicated().sum()

print(f"Duplicate Rows: {duplicate_rows}")

Duplicate Rows: 0


# Dropping Columns with 100% missing values

In [38]:
columns_to_drop = [
    "host_verifications",
    "calendar_updated",
    "host_total_listings_count",
    "neighborhood_overview",
    "host_response_rate",
    "instant_bookable",
    "host_response_time"
]

listings_clean = listings_clean.drop(columns=columns_to_drop)

#listings_clean = listings_clean.drop(columns=columns_to_drop)
listings_clean.shape

(30259, 78)

In [32]:
# Clean percentage columns

if "host_response_rate" in listings_clean.columns:

    listings_clean["host_response_rate"] = (
        listings_clean["host_response_rate"]
        .astype(str)
        .str.replace("%", "", regex=False)
    )

    listings_clean["host_response_rate"] = pd.to_numeric(
        listings_clean["host_response_rate"],
        errors="coerce"
    )


In [33]:
#Clean Price

listings_clean["price"] = (
    listings_clean["price"]
    .replace(r"[$,]", "", regex=True)
)

listings_clean["price"] = pd.to_numeric(
    listings_clean["price"],
    errors="coerce"
)

In [34]:
# Fill reviews with 0 instead of null

listings_clean["reviews_per_month"] = (
    listings_clean["reviews_per_month"]
    .fillna(0)
)

In [35]:
#convert t and f into true and false

boolean_columns = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "instant_bookable"
]

for column in boolean_columns:
    if column in listings_clean.columns:
        listings_clean[column]=(
            listings_clean[column].map({
                "t":True,
                "f":False
            })
        )

In [36]:
#Convert date columns

date_columns=["first_review", "last_review"]

for column in date_columns:
    if column in listings_clean.columns:
        listings_clean[column]=pd.to_datetime(
            listings_clean[column], errors="coerce"
        )

In [39]:
# Verify missing values again
missing_report_after = pd.DataFrame({
    "Missing Values": listings_clean.isnull().sum(),
    "Missing Percentage":
        (
            listings_clean.isnull().sum()
            /
            len(listings_clean)
        ) * 100
})

missing_report_after = missing_report_after.sort_values(
    by="Missing Percentage",
    ascending=False
)

missing_report_after.head(20)

,Missing Values,Missing Percentage
license,24973,82.530817
host_about,12410,41.012591
bedrooms,10973,36.263591
bathrooms,10112,33.418157
beds,9420,31.131234
price_quote_total_price,8745,28.900492
price_quote_price_per_night,8745,28.900492
price,8744,28.897188
estimated_revenue_l365d,8744,28.897188
review_scores_value,8568,28.315542


In [40]:
# save the cleaned dataset
# Save the cleaned dataset

listings_clean.to_csv(
    "../data/clean/listings_clean.csv",
    index=False
)

print("✅ Clean dataset saved successfully!")

✅ Clean dataset saved successfully!
